# Introduction to Diffusion Models
This notebook gives you an introduction to diffusion models, a technical intuition of their inner workings, and an overview of the individual parts that are needed to make them work. We mostly spend time on developing a mathematical intuition of diffusion, because this is where the magic happens. The choice of the model architecture, while important, is secondary to why diffusion works so well as it does.


Diffusion models are a collection of architectures and procedures that are fundamentally based on the idea of modeling the reverse function of the diffusion process and its foundation was laid in the paper [Deep Unsupervised Learning using Nonequilibrium Thermodynamics](https://arxiv.org/abs/1503.03585). Probably, this didn't really help you understading diffusion models any better, so let's start from the very basics.  

Diffusion models, unlike other generative models like Generative Adversarial Networks and Variational Auto-Encoders, don't generate an image in one inference step. Instead, they start with pure Gaussian noise in the shape of the image and remove over several steps some of the noise from which at the end the final image emerges. 


Learning to generate an image in one step, or anything non-trivial as a matter of fact, is a difficult task to learn. For this reason diffusion models break it down into a much more managable task. To understand this, let's take a look at physical diffusion, the process that inpired this model family, in the image below. When we put a drop of ink inside a glass of water, we can observe that it diffuses over time until the liguid becomes homogeneous. Imagine you would have to predict how the initial drop looked like from most right, homogeneous state. Pretty difficult, right? It would be much easier to infere the original drop from the state that right after it. This is the idea behind the step-by-step denoising process of diffusion models. 

![image](../img/diffusion_process_ink.png)

We can take this process as an analogy for the image generation for which we want to train our model. Imagine the initial state where the drop just entered the water as our image we want to generate and the homogeneous end state as the pure noise from which we start generating our image from. This process can be modeled as a Markov chain where we define a forward diffusion process $q(x_t | x_{t-1})$ that adds a bit of noise at each timestep until the image is pure noise at the final timestep. This sequence of noisy data at different timesteps is what we eventually use to train our model to **reverse** the diffusion process, hence why we call our learned $p_\theta(x_{t-1}|x_t)$ the reverse diffusion process (in case you aren't aquainted with the mathematical notation of machine learning papers, when you see a function with a subscript theta, like $f_\theta$, it means that this is a learned function, i.e. your neural network. $\theta$ is widely used to represent the learnable parameters). You may ask yourself, how we determine the amount of noise added by $q(x_t | x_{t-1})$ at each timestep. We are actually free to define that ourselves and depending on our choices the model will become better or worse at certain things. It's not that important to know all the details for understanding how diffusion models work at a high level, but if you want to dive deeper into noise schdules, check out `notebooks/noise_scheduling.ipynb`. 

![image](../img/diffusion_process_ho_2020.png)


The general problem when doing ML for generative tasks, is to build a model that is flexible enough to represent the richness of the data distrbution it's supposed to learn while still being tractable, i.e. computationally efficient. It's easy to see that diffusion is a great fit for this for many reasons. Because the basic mechanism is build on the step-wise destruction of the structure in the data distribution, but isn't tied to a medium, you can easily modify the model to suit other modalities than images. There is a swath of different applications for diffusion models outside of image generation, even including LLMs. Check out the table of publications below to see how versatile diffusion is.


| Application | Publication |
| --- | --- |
| Sound |[DiffWave: A Versatile Diffusion Model for Audio Synthesis](https://arxiv.org/abs/2009.09761) |
| Timeseries |[Conditional Score-based Diffusion Models for Probabilistic Time Series Imputation](https://arxiv.org/abs/2107.03502)|
| Planning in robotics | [Planning with diffusion for flexible behavior synthesis](https://arxiv.org/abs/2205.09991)|
| Protein synthesis |[De novo design of protein structure and function with RFdiffusion](https://www.nature.com/articles/s41586-023-06415-8)|
| Text |[Simple and Effective Masked Diffusion Language Models](https://arxiv.org/abs/2406.07524)|

A short disclaimer about the notation: Some of the notation in those notebooks is simplified or different to the original papers in service of consistency and readbability. For an accurate descriptions of the corresponding notations, please follow the links to the papers provided.

In [ ]:
from sklearn import datasets
import numpy as np
import plotly.graph_objects as go

In [ ]:
figure_size = 600

## Example of a Diffusion Process
You are probably used to hear about diffusion being used to generate images. But for the sake of educational value, we use point clouds to illustrate how diffusion works in this notebook. While we add white noise to the pixels of an image to destroy the information, in the point cloud, we add noise to the positions of the individual points. In both cases, we destroy the structure of the orginal state and want to figure out how we can reverse it.


Below you find an interactive animation of the diffusion process applied to a point cloud forming a swiss roll. Play around with the slider and see how it affects the distribution of the points. With the slider you're adjusting $\gamma$ (sometimes also called $\overline{\alpha}$ for legacy reasons). You can imagine it as a signal retention factor, where $\gamma = 1$ means that you have clean data and $\gamma = 0$ is pure noise. The transition from $\gamma: 1 \rightarrow 0$ representing forward diffusion and $\gamma: 0 \rightarrow 1$ the reverse diffusion process. You will find more details about how noise scheduling works in `notebooks/noise_scheduling.ipynb`.

In [ ]:
points, _ = datasets.make_swiss_roll(1000, noise=0.0)
points = np.concat([points[:, :1], points[:, 2:]], axis=1)
points = (points - points.mean()) / points.std()
noise = np.random.randn(*points.shape)

max_abs_lim = 0

gammas = 1 - np.logspace(0, -4, 200)
gammas = np.append(gammas, 1)
all_data = []
for gamma in gammas:
    p = np.sqrt(gamma) * points + np.sqrt(1 - gamma) * noise
    max_abs_lim = np.max([np.abs(p).max(), max_abs_lim])
    all_data.append(go.Scattergl(
        x=p[:, 0], 
        y=p[:, 1], 
        mode='markers',
        visible=(gamma == 0.0)
    ))

fig = go.Figure(data=all_data)

steps = []
for i, gamma in enumerate(gammas):
    step = dict(
        method="update",
        args=[{"visible": [j == i for j in range(len(gammas))]}],
        label=f"{gamma:.4f}"
    )
    steps.append(step)

sliders = [dict(active=0, 
                steps=steps, 
                currentvalue=dict(prefix="Gamma: ", visible=True, xanchor="center"),
            )]

ax_lim = [-np.ceil(max_abs_lim), np.ceil(max_abs_lim)]
fig.update_layout(
    title=dict(
        text="Diffusion Process",
        x=0.5,  # Center horizontally (0=left, 1=right)
        xanchor='center',
    ),
    width=figure_size,
    height=figure_size,
    xaxis={"range": ax_lim, "dtick": 1, "constrain": "domain"},
    yaxis={"range": ax_lim, "dtick": 1, "scaleanchor": "x", "scaleratio": 1, "constrain": "domain"},
    sliders=sliders
)

fig.show()

## A Selection of Different Modeling Approaches
After the last demo, you are probably curious how we actually model our denoising process. This is the point where different implementations differ. Although the general diffusion mechanism always stays the same, namely make a prediction that allows you to estimate the next timestep's state, there are different variations what part we let the model infer. The original model was predicting the next timestep's probability distribution. In this repository, though, we will do noise prediction. 

There are more things that differ between diffusion approaches, but the most important ones are what you learn/predict, how do you use your prediction to sample the next step, and how do you define the noising process. More often than not, the latter two are tuned to the choice of prediction. The table below that gives you a selection of different approaches.

| Name | What it predicts | Publication |
| --- | --- | --- |
|Diffusion probabilistic models| $\mu_{t-1}$, and $\Sigma_{t-1}$, the mean and Variance of the next timestep's Gaussian probablility distribution | [Deep Unsupervised Learning using Nonequilibrium Thermodynamics](https://arxiv.org/abs/1503.03585) |
| Denoising diffusion probabilistic models (DDPMs) and denoising diffusion implicit models (DDIMs) | $\epsilon(t)$, the noise at the current timestep | [Denoising Diffusion Probabilistic Models](https://arxiv.org/abs/2006.11239), [Denoising Diffusion Implicit Models](http://arxiv.org/abs/2010.02502) |
|Score-based models|$\nabla_x \log p_t(x)$, the time-dependent gradient field of the log probability of the perturbed data disribution, a.k.a. "Score"| [Score-Based Generative Modeling through Stochastic Differential Equations](https://arxiv.org/abs/2011.13456)|
|Flow matching models| $v(x, t)$, the velocity field, a time-dependent vector field, that defines the flow from noise to data| [Flow Matching for Generative Modeling](http://arxiv.org/abs/2210.02747) |

One of the best resources to understand the connection between DDIMs and Flow Matching is this blog post: [Diffusion Meets Flow Matching: Two Sides of the Same Coin](https://diffusionflow.github.io). Give it a read if you want to have a more in-depth comparison of the concepts.

In the following sections, we explain and visualize what part the different modeling approaches predict and how it is used to estimate the next timestep's state. 

## Diffusion Probabilistic Models
The original paper describing diffusion probabilistic models, [Deep Unsupervised Learning using Nonequilibrium Thermodynamics](https://arxiv.org/abs/1503.03585), was built pretty close around the ideas of the Markovian process that was described in the previous section. The forward process applies Gaussian noise to the data according to the variance schedule $\beta$ at every step. Since $q(x_t | x_{t-1})$ is a Gaussian distribution and $\beta_t$ is small, then we can assume the reverse function, $q(x_{t-1} | x_t)$, to be a Gaussian distribution as well. For the reverse process, we learn to predict the mean and variance of the previous step with $f_\mu(x_t, t)$ and $f_\Sigma(x_t)$ respectively and use them to sample the previous step $x_{t-1} \sim N(\mu_{t-1}, \Sigma_{t-1})$. Because of the assumption that $\beta$ is small, the reverse process of diffusion probabilistic models is done over a large amount of timesteps. In practice, this number is usually around 1000 timesteps per generation.

## Denoising Diffusion Probabilistic Models
In the paper [Denoising Diffusion Probabilistic Models](https://arxiv.org/abs/2006.11239) the authors reformulate the objective as denoising of the current timestep. By doing this, they simplify both training and sampling vastly, since the model no longer learns the transition between states in a Markovian process but rather the predicting the noise in $x_t$. Algorithm 1 and 2 in the paper (also shown below) show the elegant and straightforward nature of this approach quite nicely.

![image](../img/training_sampling_algorithm_ho_2020.png)

## Denoising Diffusion Implicit Models
While the last paper was simplifying the formulation of the diffusion process and training objective significanlty, the contribution of [Denoising Diffusion Implicit Models](http://arxiv.org/abs/2010.02502) is extending the formulation of the problem in a way that allows to skip certain steps, i.e. making the denoising process non-Markovian. This sounds more complicated than it actually is. Practically everything stays the same, except for the sampling, where instead of calculating $x_{t-1}$, we directly go to $x_{t-N}$ (with the value of $N$ depending on how aggressively we want to skip steps). 


One thing that all three implementations have in common, though. Even with a perfect understanding of the noise in the previous step, the reverse process still follows a suboptimal trajectory, which is curved instead of straight to the noise-free position. This is because the reverse process is following the noise trajectory of the forward process which is curved due to the measures for variance preservation. Take a look at the plot below. We picked some points and follow them through the diffusion process. The blue line is the noise, i.e. the delta between the current and the noise-free position of the point, the red arrow points in the direction of the next state, towards that the reverse diffusion process will go, and the orange line is the (de-)noising trajectory of the corresponding point. 

In [ ]:
points, _ = datasets.make_swiss_roll(1000, noise=0.0)
points = np.concat([points[:, :1], points[:, 2:]], axis=1)
points = (points - points.mean()) / points.std()
noise = np.random.randn(*points.shape)

# Pick 5 random points to highlight
n_highlight = 5
highlight_indices = np.random.choice(len(points), n_highlight, replace=False)
highlight_mask = np.isin(np.arange(len(points)), highlight_indices)

max_abs_lim = 0
gammas = 1 - np.logspace(0, -4, 200)
gammas = np.append(gammas, 1)
frame_data = []
all_data = []

for gamma in gammas:
    p = np.sqrt(gamma) * points + np.sqrt(1 - gamma) * noise
    max_abs_lim = np.max([np.abs(p).max(), max_abs_lim])
    frame_data.append(p)

# Dims are (point, timestep, coord)
point_trajectories = np.stack(frame_data).transpose(1, 0, 2)    

opacities = np.where(highlight_mask, 1.0, 0.8)
colors = np.where(highlight_mask, 1, 0)  # Use numeric for color scale

for frame_nr in range(len(frame_data)):
    # Precompute marker properties with numpy
    # Single combined trace
    all_data.append(go.Scattergl(
        x=frame_data[frame_nr][:, 0],
        y=frame_data[frame_nr][:, 1],
        mode='markers',
        marker=dict(
            color=colors, 
            opacity=opacities,
            colorscale=[[0, 'lightgray'], [1, 'red']],
            showscale=False
        ),
        visible=(frame_nr == 0),
        showlegend=False
    ))
    
    # Error vectors as separate line trace
    error_x = []
    error_y = []
    for idx in highlight_indices:
        clean = points[idx]
        current = frame_data[frame_nr][idx]
        # None is a line break in Plotly
        error_x.extend([clean[0], current[0], None])
        error_y.extend([clean[1], current[1], None])
    
    all_data.append(go.Scattergl(
        x=error_x,
        y=error_y,
        mode='lines',
        line=dict(color='blue', width=1),
        visible=(frame_nr == 0),
        showlegend=False
    ))

    # Create point trajectories
    point_traj_x = []
    point_traj_y = []

    for idx in highlight_indices:
        point_traj_x.extend(point_trajectories[idx, :frame_nr+1, 0].tolist())
        point_traj_x.append(None)
        point_traj_y.extend(point_trajectories[idx, :frame_nr+1, 1].tolist())
        point_traj_y.append(None)

    all_data.append(go.Scatter(  # Not Scattergl
        x=point_traj_x, 
        y=point_traj_y, 
        mode='lines',
        line=dict(color='orange', width=2),
        visible=(frame_nr == 0),
        showlegend=False
    ))

    # Create lines pointing in the direction that the point will progress
    frame_data_idx = frame_nr 
    if frame_nr + 1 == len(frame_data):
        # Just use the same direction verctor again as before at the last step
        frame_data_idx = frame_nr - 1

    traj_x = []
    traj_y = []
    for idx in highlight_indices:
        current = frame_data[frame_data_idx][idx]
        next = frame_data[frame_data_idx+1][idx]
        delta = (next - current)
        delta_length = np.sqrt(delta[0]**2 + delta[1]**2)
        normalized_delta = delta / delta_length
        target = current + 0.5 * normalized_delta
        traj_x.extend([current[0], target[0], None])
        traj_y.extend([current[1], target[1], None])

    all_data.append(go.Scatter(  # Not Scattergl
        x=traj_x, 
        y=traj_y, 
        mode='lines+markers',
        line=dict(color='firebrick', width=2),
        marker=dict(
            size=15,
            symbol='arrow',
            angleref='previous'
        ),
        visible=(frame_nr == 0),
        showlegend=False
    ))
    

fig = go.Figure(data=all_data)

steps = []
traces_per_frame = 4
for i, gamma in enumerate(gammas):
    step = dict(
        method="update",
        args=[{"visible": [j // traces_per_frame == i for j in range(len(all_data))]}],
        label=f"{gamma:.4f}"
    )
    steps.append(step)

sliders = [dict(
    active=0, 
    steps=steps, 
    currentvalue=dict(prefix="Gamma: ", visible=True, xanchor="center")
)]

ax_lim = [-np.ceil(max_abs_lim), np.ceil(max_abs_lim)]
fig.update_layout(
    title=dict(
        text="DDPM Reverse Diffusion Process",
        x=0.5,  # Center horizontally (0=left, 1=right)
        xanchor='center',
    ),
    width=figure_size,
    height=figure_size,
    xaxis={"range": ax_lim, "dtick": 1, "constrain": "domain"},
    yaxis={"range": ax_lim, "dtick": 1, "scaleanchor": "x", "scaleratio": 1, "constrain": "domain"},
    sliders=sliders
)
fig.show()

## Flow Matching Models
Flow matching models approache the generation process from a completely different mathematical perspective. They model the convertion of noise to an output as a optimal transition of one distribution to another. The mathematical field that is concerned with this kind of topic is called **Optimal Transport**. The orginal paper, [Flow Matching for Generative Modeling](http://arxiv.org/abs/2210.02747), analyzes the suboptimality of the denoising trajectory of diffusion models and presents an approach of modeling a vector field that leads to optimal denoising trajectories. The idea is that instead of learning to predict the noise on the input, we learn to predict the direction in which we have to move the the values of the noised input to come closer to the target distribution, in our case the denoised image. This vector field has as many dimensions as our data (in our case 2000, with 1000 dots with two coordinate values each), hence is difficult to visualize in it's entirety. For this reason, I visualized it for some single points in the figures below. But take the example below with a grain of salt, the vector field is time-dependent and can look different at different points in time. A way to understand this intuitively is by thinking about how strong you would have to move a point far away from its target position depending where you are on the time axis: if there is still a lot of time left, you don't have to move that quickly compared to when you are close to the end. 


In [ ]:
from plotly.subplots import make_subplots

# Generate swiss roll without noise
points, _ = datasets.make_swiss_roll(1000, noise=0.0, random_state=42)
points = np.concat([points[:, :1], points[:, 2:]], axis=1)
points = (points - points.mean()) / points.std()

# Pick 2 different random points to highlight
highlight_point_1 = points[1]
highlight_point_2 = points[3]

# Create a grid for the quiver plot (vector field)
ax_lim = (-4, 4)
n_meshgrid = 9
meshgrid_range = np.linspace(ax_lim[0], ax_lim[1], n_meshgrid)
X, Y = np.meshgrid(meshgrid_range, meshgrid_range)

# Create vector fields for both points
U1 = highlight_point_1[0] - X
V1 = highlight_point_1[1] - Y

U2 = highlight_point_2[0] - X
V2 = highlight_point_2[1] - Y

# Scale vectors for better visualization
scaling_factor = 0.5
U1_norm = scaling_factor * U1
V1_norm = scaling_factor * V1
U2_norm = scaling_factor * U2
V2_norm = scaling_factor * V2

# Create subplots (1 row, 2 columns)
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Sample 1", "Sample 2"),
    horizontal_spacing=0.1
)

# First subplot
# Add scatter plot of all points
fig.add_trace(go.Scattergl(
    x=points[:, 0],
    y=points[:, 1],
    mode='markers',
    marker=dict(color='lightgray', size=3),
    showlegend=False,
    hoverinfo='skip'
), row=1, col=1)

# Add highlighted point 1
fig.add_trace(go.Scattergl(
    x=[highlight_point_1[0]],
    y=[highlight_point_1[1]],
    mode='markers',
    marker=dict(color='red', size=10),
    showlegend=False
), row=1, col=1)

# Add quiver plot for point 1
for i in range(len(meshgrid_range)):
    for j in range(len(meshgrid_range)):
        fig.add_trace(go.Scatter(
            x=[X[j, i], X[j, i] + 0.3 * U1_norm[j, i]],
            y=[Y[j, i], Y[j, i] + 0.3 * V1_norm[j, i]],
            mode='lines+markers',
            line=dict(color='blue', width=1.5),
            marker=dict(size=6, symbol='arrow', angleref='previous'),
            showlegend=False,
            hoverinfo='skip'
        ), row=1, col=1)

# Second subplot
# Add scatter plot of all points
fig.add_trace(go.Scattergl(
    x=points[:, 0],
    y=points[:, 1],
    mode='markers',
    marker=dict(color='lightgray', size=3),
    showlegend=False,
    hoverinfo='skip'
), row=1, col=2)

# Add highlighted point 2
fig.add_trace(go.Scattergl(
    x=[highlight_point_2[0]],
    y=[highlight_point_2[1]],
    mode='markers',
    marker=dict(color='red', size=10),
    showlegend=False
), row=1, col=2)

# Add quiver plot for point 2
for i in range(len(meshgrid_range)):
    for j in range(len(meshgrid_range)):
        fig.add_trace(go.Scatter(
            x=[X[j, i], X[j, i] + 0.3 * U2_norm[j, i]],
            y=[Y[j, i], Y[j, i] + 0.3 * V2_norm[j, i]],
            mode='lines+markers',
            line=dict(color='blue', width=1.5),
            marker=dict(size=6, symbol='arrow', angleref='previous'),
            showlegend=False,
            hoverinfo='skip'
        ), row=1, col=2)

# Update layout
fig.update_layout(
    title=dict(
        text="Flow-Generating Vector Field for Different Samples",
        x=0.5,
        xanchor='center',
    ),
    width=figure_size * 2,
    height=figure_size,
    showlegend=False
)

# Update both x and y axes for both subplots
fig.update_xaxes(range=ax_lim, dtick=1, constrain='domain', row=1, col=1)
fig.update_yaxes(range=ax_lim, dtick=1, scaleanchor='x', scaleratio=1, constrain='domain', row=1, col=1)
fig.update_xaxes(range=ax_lim, dtick=1, constrain='domain', row=1, col=2)
fig.update_yaxes(range=ax_lim, dtick=1, scaleanchor='x2', scaleratio=1, constrain='domain', row=1, col=2)

fig.show()

I bet now you can see that denoising in the direction of the vector field leads to a more efficient trajectory. Interestingly, one would come to a similar result of the if one would sample DDIMs with simple Euler, i.e. estimating the noise and then using its direction as gradient to follow step-wise. The problem with this is that it violates the variance-preserving condition that they defined in their (de-)noising processes which would lead to a different input data distribution during inference than it was trained and could cause a drop in performance (despite of it, Simple Euler is used with success in some cases). The flow matching paper, on the other hand, abandons this condition and defines the mean and  standard deviation as linearly changing in time.

$$
x_t \sim N(\mu_t,\sigma_t I ) \text{ with } \mu_t = tx_1, \sigma_t = 1 - (1-\sigma_{\text{min}})t 
$$

where $x_t$ is the noised sample at time $t \in [0,1]$, $x_1$ the target sample ($x_0$ is pure noise), and $\sigma_{\text{min}}$ a sufficiently small standard deviation (in practice, you can choose the average standard deviation of your samples). 


Check out the figure below to get a better intuition what this means for the denoising trajectories.


In [ ]:
points, _ = datasets.make_swiss_roll(1000, noise=0.0)
points = np.concat([points[:, :1], points[:, 2:]], axis=1)
points = (points - points.mean()) / points.std()
noise = np.random.randn(*points.shape)

# Pick 5 random points to highlight
n_highlight = 5
highlight_indices = np.random.choice(len(points), n_highlight, replace=False)
highlight_mask = np.isin(np.arange(len(points)), highlight_indices)

max_abs_lim = 0
timesteps = np.linspace(0, 1, 201)
frame_data = []
all_data = []

for t in timesteps:
    # Assuming u(t) = points - current_pos, we can derive the positions
    d = 2 * (t - t**2/2)
    p = d * points + (1 - d) * noise 
    max_abs_lim = np.max([np.abs(p).max(), max_abs_lim])
    frame_data.append(p)


# Dims are (point, timestep, coord)
point_trajectories = np.stack(frame_data).transpose(1, 0, 2)    

opacities = np.where(highlight_mask, 1.0, 0.8)
colors = np.where(highlight_mask, 1, 0)  # Use numeric for color scale

for frame_nr in range(len(frame_data)):
    # Precompute marker properties with numpy
    # Single combined trace
    all_data.append(go.Scattergl(
        x=frame_data[frame_nr][:, 0],
        y=frame_data[frame_nr][:, 1],
        mode='markers',
        marker=dict(
            color=colors, 
            opacity=opacities,
            colorscale=[[0, 'lightgray'], [1, 'red']],
            showscale=False
        ),
        visible=(frame_nr == 0),
        showlegend=False
    ))
    
    # Error vectors as separate line trace
    error_x = []
    error_y = []
    for idx in highlight_indices:
        clean = points[idx]
        current = frame_data[frame_nr][idx]
        # None is a line break in Plotly
        error_x.extend([clean[0], current[0], None])
        error_y.extend([clean[1], current[1], None])
    
    all_data.append(go.Scattergl(
        x=error_x,
        y=error_y,
        mode='lines',
        line=dict(color='blue', width=1),
        visible=(frame_nr == 0),
        showlegend=False
    ))

    # Create point trajectories
    point_traj_x = []
    point_traj_y = []

    for idx in highlight_indices:
        point_traj_x.extend(point_trajectories[idx, :frame_nr+1, 0].tolist())
        point_traj_x.append(None)
        point_traj_y.extend(point_trajectories[idx, :frame_nr+1, 1].tolist())
        point_traj_y.append(None)

    all_data.append(go.Scatter(  # Not Scattergl
        x=point_traj_x, 
        y=point_traj_y, 
        mode='lines',
        line=dict(color='orange', width=2),
        visible=(frame_nr == 0),
        showlegend=False
    ))

    # Create lines pointing in the direction that the point will progress
    frame_data_idx = frame_nr 
    if frame_nr + 1 == len(frame_data):
        # Just use the same direction verctor again as before at the last step
        frame_data_idx = frame_nr - 1

    traj_x = []
    traj_y = []
    for idx in highlight_indices:
        current = frame_data[frame_data_idx][idx]
        next = frame_data[frame_data_idx+1][idx]
        delta = (next - current)
        delta_length = np.sqrt(delta[0]**2 + delta[1]**2)
        normalized_delta = delta / delta_length
        target = current + 0.5 * normalized_delta
        traj_x.extend([current[0], target[0], None])
        traj_y.extend([current[1], target[1], None])

    all_data.append(go.Scatter(  # Not Scattergl
        x=traj_x, 
        y=traj_y, 
        mode='lines+markers',
        line=dict(color='firebrick', width=2),
        marker=dict(
            size=15,
            symbol='arrow',
            angleref='previous'
        ),
        visible=(frame_nr == 0),
        showlegend=False
    ))
    

fig = go.Figure(data=all_data)

steps = []
traces_per_frame = 4
for i, gamma in enumerate(gammas):
    step = dict(
        method="update",
        args=[{"visible": [j // traces_per_frame == i for j in range(len(all_data))]}],
        label=f"{gamma:.4f}"
    )
    steps.append(step)

sliders = [dict(
    active=0, 
    steps=steps, 
    currentvalue=dict(prefix="Time (normalized): ", visible=True, xanchor="center")
)]

ax_lim = [-np.ceil(max_abs_lim), np.ceil(max_abs_lim)]
fig.update_layout(
    title=dict(
        text="Flow Matching Diffusion Reverse Process",
        x=0.5,  # Center horizontally (0=left, 1=right)
        xanchor='center',
    ),
    width=figure_size,
    height=figure_size,
    xaxis={"range": ax_lim, "dtick": 1, "constrain": "domain"},
    yaxis={"range": ax_lim, "dtick": 1, "scaleanchor": "x", "scaleratio": 1, "constrain": "domain"},
    sliders=sliders
)
fig.show()

# Key Takeaways
In this notebook, you received a general overview of what diffusion im ML is and which different modeling approaches exist. We checked out how the different implementations impacted the denoising process and how they were improved upon.

- Diffusion is more than a model architecture, it's a mathematical framework to model generative ML tasks
- The three major parts of each diffusion implementation are the (de-)noising schedule, choice of prediction, and sampling method
- Diffusion models are not constrained to images. Its data-agnostic mathematical framework allows it to be applied to any kind of data

Now that you understand the basic mathematical framework of diffusion, it's a great time to dive deeper on any of the topics covered in the notebooks. You can also start training your own model by following the README and gain some practical experience.
